# Lab 00 — Introduction and Setup

**Deep Learning for Computer Vision · AIT**

Two goals for today:

1. Get a working environment — **Google Colab is the recommended path** and every lab in
   this course is written to run on its free T4 GPU.
2. See what computer vision can do in 2026, before we spend ten weeks taking it apart.

---

## Which environment should I use?

| | Use this if | Notes |
|---|---|---|
| **Google Colab (recommended)** | you have no NVIDIA GPU, or you want zero setup | Free T4, 16GB. Every lab is sized to fit. Runtime → Change runtime type → **T4 GPU** |
| **Your own laptop** | you have an NVIDIA GTX/RTX GPU | Use miniconda. See the appendix at the bottom |

> **Colab caveat.** Free Colab disconnects after a period of inactivity and has a session
> time limit. Every lab in this course saves checkpoints for exactly this reason — mount
> your Google Drive and save there, or you will retrain from scratch.

Apple Silicon (M-series) works for most labs via the `mps` backend, but a few operations
fall back to CPU and the generative labs (09, 10) will be slow. Colab is easier.

## Verify your environment

Run the cells below. If the GPU check fails on Colab, go to
**Runtime → Change runtime type → Hardware accelerator → T4 GPU** and re-run.

In [ ]:
import sys, platform
print("Python  :", sys.version.split()[0])
print("Platform:", platform.system(), platform.release())
assert sys.version_info >= (3, 9), "This course needs Python 3.9 or newer"
print("OK")

In [ ]:
import torch
print("PyTorch :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("Memory  : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    cc = torch.cuda.get_device_capability(0)
    print("Compute capability: %d.%d" % cc)
    if cc[0] < 8:
        print("\n  NOTE: this GPU is pre-Ampere (a T4 is 7.5).")
        print("  -> use torch.float16 for mixed precision, NOT bfloat16")
        print("  -> FlashAttention-2 is unavailable; use attn_implementation='sdpa'")
else:
    print("\n  No GPU detected. Labs will run but slowly.")
    print("  On Colab: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# A 5-second benchmark so you know what your hardware is worth.
import time, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
a = torch.randn(4096, 4096, device=device)
b = torch.randn(4096, 4096, device=device)

for _ in range(3):                       # warm-up
    _ = a @ b
if device == "cuda":
    torch.cuda.synchronize()

t0 = time.time()
for _ in range(20):
    _ = a @ b
if device == "cuda":
    torch.cuda.synchronize()
dt = time.time() - t0

print("20 x (4096^3 matmul) in %.2fs  ->  %.1f TFLOP/s" % (dt, 20 * 2 * 4096 ** 3 / dt / 1e12))

## Mount Google Drive (Colab users)

Do this at the start of every lab that trains something. Checkpoints go to Drive so a
disconnect does not cost you an hour.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    CKPT_DIR = "/content/drive/MyDrive/dlcv_checkpoints"
    os.makedirs(CKPT_DIR, exist_ok=True)
    print("checkpoints ->", CKPT_DIR)
except ImportError:
    CKPT_DIR = "./checkpoints"
    import os
    os.makedirs(CKPT_DIR, exist_ok=True)
    print("not on Colab; checkpoints ->", CKPT_DIR)

---
# What computer vision can do in 2026

Four demos, four labs they map onto. Run them, then forget the details — we build all of
this from the ground up over the next ten weeks.

In [ ]:
!pip install -q ultralytics transformers diffusers accelerate

In [ ]:
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

def load_image(url):
    return Image.open(BytesIO(requests.get(url, timeout=20).content)).convert("RGB")

img = load_image("https://ultralytics.com/images/bus.jpg")
plt.figure(figsize=(5, 7)); plt.imshow(img); plt.axis("off"); plt.title("our test image"); plt.show()

### 1. Object detection — *Lab 07*

In [ ]:
from ultralytics import YOLO

det = YOLO("yolo11n.pt")
r = det.predict(img, conf=0.4, verbose=False)[0]

print("found:", [r.names[int(c)] for c in r.boxes.cls])
plt.figure(figsize=(5, 7)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off")
plt.title("YOLO11 - boxes and labels"); plt.show()

### 2. Segmentation — *Lab 08*

In [ ]:
seg = YOLO("yolo11n-seg.pt")
rs = seg.predict(img, conf=0.4, verbose=False)[0]

plt.figure(figsize=(5, 7)); plt.imshow(rs.plot()[:, :, ::-1]); plt.axis("off")
plt.title("pixel-accurate masks, not just boxes"); plt.show()

### 3. Asking questions about an image — *Lab 11*

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

proc = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-256M-Instruct")
vlm = AutoModelForVision2Seq.from_pretrained(
    "HuggingFaceTB/SmolVLM-256M-Instruct", torch_dtype=dtype).to(device).eval()

msg = [{"role": "user", "content": [{"type": "image"},
                                    {"type": "text", "text": "Describe this scene in one sentence."}]}]
prompt = proc.apply_chat_template(msg, add_generation_prompt=True)
inputs = proc(text=prompt, images=[img], return_tensors="pt").to(device, dtype)

with torch.no_grad():
    out = vlm.generate(**inputs, max_new_tokens=60, do_sample=False)
print(proc.batch_decode(out, skip_special_tokens=True)[0].split("Assistant:")[-1].strip())

### 4. Generating an image from text — *Labs 09 and 10*

In [ ]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=dtype,
    safety_checker=None, requires_safety_checker=False).to(device)
pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()

out = pipe("a watercolour painting of Bangkok at dusk, warm light",
           num_inference_steps=25, guidance_scale=7.5,
           generator=torch.Generator(device=device).manual_seed(0)).images[0]
plt.figure(figsize=(5, 5)); plt.imshow(out); plt.axis("off")
plt.title("Stable Diffusion - from pure noise"); plt.show()

---
## What is coming

| Lab | Topic | You will build |
|---|---|---|
| 01 | Python & NumPy | Python and basic libraries |
| 02 | Neural networks from scratch | backprop in NumPy, no frameworks |
| 03 | CNNs and PyTorch | convolution by hand, then a real CNN |
| 04 | CNN architectures | ResNet, SE blocks, ConvNeXt |
| 05 | Transfer learning & evaluation | the workflow you will use in every project |
| 06 | Transformers & ViT | self-attention, from scratch |
| 07 | Object detection | IoU, NMS, mAP by hand; YOLO11; DETR |
| 08 | Segmentation | U-Net from scratch; SAM |
| 09 | Diffusion models | DDPM, DDIM and guidance, from scratch |
| 10 | Stable Diffusion & LoRA | fine-tune a generator on your own images |
| 11 | CLIP & VLMs | a working mini-LLaVA |

Everything you ran above is built by hand somewhere in that list.

---
## Getting the course materials

```bash
git clone https://github.com/l-kuo/DLCV-Lab.git
cd DLCV-Lab
```

On Colab, add this at the top of any lab notebook instead:

```python
!git clone https://github.com/l-kuo/DLCV-Lab.git
%cd DLCV-Lab/DLCV-04-CNN-Architectures
```

---
# Appendix — detailed OS-by-OS installation

The instructions below cover installing Python, conda and git on Windows, Linux and
macOS, plus how to embed screenshots in a Jupyter markdown cell. You only need this if
you are setting up a local machine rather than using Colab.


## Setup  
When you start the setup from the scratch with any of Windows, Linux or Mac OS, you are recommended to install the following:  
- Python3 Latest (required)
- Conda Environment (miniconda: recommended)
- PIP (required even if you are using conda environment)
- Python Libraries
- Git Installation

### Python3 Installation  
- If you have Nvidia GTX or RTX GPU built-in your laptop, you can use your **Own Laptop** throughout the course.  
- If you don't have Nvidia GTX or RTX GPU in your laptop, it is recommended to use your **Google Colab** instead since GPU processing is required in most deep learning trainings.  
- We will use **Python3** throughout the lab. If you have lower versions of Python such as Python2, you have to install Python3 since a lot of syntaxes are different.  
- To install Python3 package in your machine, please follow below instructions according to your OS.

**For Windows Users**
- Check the type of your processor architecture using `systeminfo | find "System Type"`
- Go to https://www.python.org/downloads/windows/ and download the installation package (.exe file) according to your architecture type. 
- You can also check `python` or `python3` in your (conda prompt) terminal or command line before installing Python3 in your laptop. It will prompt you to the Python3 interpreter.  
- To exit from Python3 interpreter, simple type `exit` or `exit()`.

<img src="img/pythonInstaller.PNG" alt="Python" width="500px" style="float: center" />
<br clear="left" />

- While installing Python from `.exe` file, make sure you tick the add Python to PATH box or you will not be able to access Python from the windows terminal.  


<img src="img/add_path.png" alt="Add Paths" width="200px" style="float: center" />
<br clear="left" />

- Install Windows Subsystem for Linux(wsl). You can choose the linux distro by using `--name` flag. By default, wsl chooses ubuntu as the linux disto.
```
wsl --install
wsl
```

**For Linux Users**
- For Ubuntu users, you don't need to install Python as it has been built-in in OS package already.  
- In order to check whether you have python3 installed, you can check `python` or `python3` command in your terminal and it will prompt you to the Python3 interpreter as shown in below image. Otherwise, you need to install Python3.
- To exit from Python3 interpreter, simple type `Ctrl+z` or `Ctrl+d`.

<img src="img/python3.png" alt="Linux Users" width="800px" style="float: center" />
<br clear="left" />

- For installation, go to terminal and run the commands:
    - `sudo apt-get update && sudo apt-get upgrade -y`
    - `sudo apt-get install python3` or if you want to install specific version, type: `sudo apt-get install python3.xx.x`
- To install Python Build Dependencies, run the command:
    - `sudo apt-get build-dep python3`
- You can also install some useful modules by running the command:
    - `sudo apt-get install build-essential gdb lcov libbz2-dev libffi-dev libgdbm-dev liblzma-dev libncurses5-dev libreadline6-dev libsqlite3-dev libssl-dev lzma lzma-dev tk-dev uuid-dev zlib1g-dev`
- Next, you need to install PIP(the Packages Installer for Python) in order to easily install python packages. To do so, run the command `sudo apt-get install python-pip` or `sudo apt-get install python3-pip`

**For MacOS Users**
- Please follow the instruction from https://www.datacamp.com/blog/how-to-install-python.  

### Conda Installation for All Users
- To check Python version in terminal: `python --version` or `python -v`
- To install miniconda, please follow the instructions from https://docs.anaconda.com/miniconda/#miniconda-latest-installer-links and select the download file according to your operating system.
- Don't forget to add Miniconda into PATH Environment Variable. See the figure below:
<img src='img/miniconda-add-path.png' alt="Miniconda-Add-Path" width="500px" style="float: center" />
<br clear="left" /> 

- Once installed miniconda, you will be able to run conda command in the Anaconda Prompt (miniconda).
- In the Anaconda Prompt, you are directed to the conda (base) environment. If you would like to create your own custom environment, you need to first exit the default (base) environment using `conda deactivate` command.
<img src='img/deactivate-base.png' alt="Deactivate-Base" width="500px" style="float: center" />
<br clear="left" />

- After you get out of the default env, in order to create a custom env name eg.`myconda` with built-in python installed, use the command `conda create -n myConda python` or specify the version of python `conda create -n myConda python==3.12.4`.
- `conda env list` command will show all the conda environments you have in your machine. You can simply activate the environment using `conda activate EnvName` eg. `conda activate myConda`.
<img src='img/conda-activate.png' alt="Conda-Activate" width="500px" style="float: center" />
<br clear="left" />

- To check installed python packages in the terminal, please use the command `conda list` for conda.
- To install a python library or package from conda: `conda install package_name`, eg. `conda install numpy`
- To install jupyter notebook: `conda install jupyter`.
- To run jupyter notebook, open terminal and go to the directory you want to run jupyter notebook, type the command `jupyter notebook`

**CUDA Toolkit and CuDNN Installation with Conda**
- Before installing the CUDA, first check your available GPU whether it is compatible with CUDA installation. Try `wmic path win32_VideoController get name` in your terminal and check if your GPU is NVIDIA RTX or GTX Series. Mac users or the users with GPUs other than NVIDIA RTX or GTX Series are not compatible with CUDA.
<img src='img/gpu.png' alt="GPU" width="500px" style="float: center" />
<br clear="left" />

- If your machine is compatible, run `conda install -c conda-forge cudatoolkit` and then `conda install -c conda-forge cudnn` inside your conda environment. The latest version of CUDA Toolkit in conda-forge to date is cudatoolkit-11.8.0 and that of CuDNN is cudnn-9.2.1.18.

**Pytorch Installation**  
- Another important package is Pytorch. You can follow the instructions from https://pytorch.org/get-started/locally/ and install with CUDA versions(here in conda env, we have CUDA 11.8).
<img src='img/pytorch.png' alt="Pytorch" width="500px" style="float: center" />
<br clear="left" />

- Those who are not using the CUDA, you can simply use the CPU Pytorch Installation.
- Finally, in Python Interpreter, type `import torch`, then `torch.cuda.is_available()` will return `True` for the CUDA users and `False` for the CPU users.
<img src='img/cuda-available.png' alt="CUDA Available" width="500px" style="float: center" />
<br clear="left" />

**Some Other Useful Packages You Might Want to Install**
The followings are some important packages or libraries however most of them are sometimes dependencies of other packages that you have installed. But make sure that you can handle the version mismatch issue properly while installing the python libraries and dependencies.
1. numpy
2. pandas
3. matplotlib
4. seaborn
5. opencv
6. setuptools

### Git Installation

Please follow the instruction from below link. Complete until the step 3. If you want to understand more about Git, you can complete all steps but not necessary. 
It is recommended to create the Git account with your own personal email address since you will be using throughout your career even after your degree.  
https://www.freecodecamp.org/news/introduction-to-git-and-github/  

### How to Clone a Git Repository

https://git-scm.com/book/en/v2/Git-Basics-Getting-a-Git-Repository

Follow the above link and try cloning my Deep Learning for Computer Vision Lab Repository.  
https://github.com/l-kuo/DLCV-Lab  

## Get Familiar with a Computer Vision Model (Object Detection and Classification)
Now we are going to play around with a famous Object Detection Model called YOLO(You Only Look Once). Different versions of YOLO have been released from different authors. Here, we are going to use YOLOv8 Model which is easy and accurate for object detection and classification. In order to execute the YOLOv8 Model, we need to install a library called Ultralytics from https://docs.ultralytics.com/quickstart/.

- Create a file named yolov8_inference.py (.py extension) and paste the following code inside the file and save it.
- You can run the python extension file in the commandline or terminal using `python yolov8_inference.py`.
- It will show up the results like below. If you pass the video file or even livestream, the model can detect and classify the objects in real-time.

<img src='img/results.png' alt="Results" width="900px" style="float: center" />
<br clear="left" />  

#### How to Put a Screenshot in Jupyter Notebook Markdown

The following steps are the easiest way to put a screenshot in your Jupyter Markdown cell.  

##### Step1
- Add a new cell at a location where you want to put the screenshot. Notice that you have to select an existing cell before adding. A new cell will appear just below the cell you selected.

<img src='img/step1.png' alt="Step 1" width="600px" style="float: center" />
<br clear="left" />  

##### Step2
- Select the new cell and change the type of the cell by choosing "Markdown" on the task bar dropdown.

<img src='img/step2.png' alt="Step 2" width="600px" style="float: center" />
<br clear="left" />  

##### Step3
- Under the "Edit" tab, choose the "Insert Image" while selecting the cell.

<img src='img/step3.png' alt="Step 3" width="300px" style="float: center" />
<br clear="left" />  

##### Step4
- Click on "Choose File" and select your screenshot from your local directory. Then click "OK".

<img src='img/step4.png' alt="Step 4" width="600px" style="float: center" />
<br clear="left" />  

##### Step5
- When you have selected the screenshot image, the script inside the red rectangle will appear automatically. Once you run the cell, you will get the screenshot image shown in your markdown file.

<img src='img/step5.png' alt="Step 5" width="600px" style="float: center" />
<br clear="left" />  

